In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, year, month, dayofmonth, hour, to_timestamp, udf, when
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import to_timestamp

spark = SparkSession.builder \
    .appName("HajjPilgrimProfessionalPipeline") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.1.1") \
    .config("spark.eventLog.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

def get_loc(coord):
    mapping = {
        '21.4225,39.8262': 'Al-Haram', 
        '21.4055,39.8912': 'Mina', 
        '21.3540,39.9015': 'Arafat', 
        '21.3650,39.9500': 'Muzdalifah',
        '21.4025,39.7262': 'Jabal Al-Noor'
    }
    return mapping.get(coord, 'Unknown')
loc_udf = udf(get_loc, StringType())

hajj_schema = StructType([
    StructField("pilgrim_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("current_location", StringType(), True),
    StructField("heart_rate", StringType(), True),
    StructField("body_temperature", StringType(), True)
])

raw_stream_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka1:9092,kafka2:9092,kafka3:9092,localhost:9092") \
    .option("subscribe", "pilgrim-movements") \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

parsed_df = raw_stream_df \
    .selectExpr("CAST(value AS STRING) as json_payload") \
    .select(from_json(col("json_payload"), hajj_schema).alias("data")) \
    .select("data.*") \
    .withColumn("ts", to_timestamp(col("timestamp"))) \
    .withColumn("year", year(col("ts"))) \
    .withColumn("month", month(col("ts"))) \
    .withColumn("day", dayofmonth(col("ts"))) \
    .withColumn("hour", hour(col("ts"))) \
    .withColumn("heart_rate", col("heart_rate").cast(DoubleType())) \
    .withColumn("body_temperature", col("body_temperature").cast(DoubleType())) \
    .fillna({"heart_rate": 80.0, "body_temperature": 37.0}) \
    .withColumn("current_location", loc_udf(col("current_location"))) \
    .withColumn("sos_alert", when((col("heart_rate") > 120) | (col("body_temperature") > 39.0), True).otherwise(False)) \
    .dropDuplicates(["pilgrim_id", "timestamp"])

hdfs_query = parsed_df.writeStream \
    .format("parquet") \
    .partitionBy("year", "month", "day", "hour") \
    .option("path", "/user/hdfs/hajj_movements/") \
    .option("checkpointLocation", "/user/hdfs/checkpoints/hdfs_data") \
    .outputMode("append") \
    .start()


def send_to_postgres(batch_df, batch_id):
    if batch_df.count() == 0: return
    
    final_df = batch_df.select(
        col("pilgrim_id").cast("string"),
        to_timestamp(col("timestamp")).alias("timestamp"), 
        col("current_location").cast("string"),
        col("sos_alert").cast("boolean"), 
        col("heart_rate").cast("float"),      
        col("body_temperature").cast("float")
    )
    
    final_df.write \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://postgres_db:5432/hajj_db") \
        .option("dbtable", "live_pilgrims_status") \
        .option("user", "postgres") \
        .option("password", "postgres") \
        .option("driver", "org.postgresql.Driver") \
        .mode("append") \
        .save()

pg_query = parsed_df.writeStream \
    .foreachBatch(send_to_postgres) \
    .option("checkpointLocation", "/user/hdfs/checkpoints/pg_data") \
    .start()

print("🚀 Pipeline is running professionally with Enrichment & Imputation!")
spark.streams.awaitAnyTermination()

🚀 Pipeline is running professionally with Enrichment & Imputation!
